# MARS pose on a GPU: a video in, keypoints out

This notebook turns a **recorded video of two mice** into the **keypoints file** that the live-tracker project reads.
It uses the MARS top-view models (Segalin et al. 2021, eLife; license CC BY-NC 4.0, non-commercial use only).

**Honest limits**
- It is **offline**: record first, analyze here, then bring the result back. It is not live.
- MARS needs a **top view with one black and one white mouse**. It does not work on any mouse video.
- The speed on a free Colab GPU is not known yet. Step 4 measures it on 60 frames before you run the whole video.

**Steps**
1. Menu **Runtime > Change runtime type > T4 GPU**, then Save.
2. Run the cells from the top, one after the other.
3. In step 3 you choose your video file. Nothing is uploaded before that.
4. At the end, download `keypoints.parquet` and `stretched.mp4`.
5. In the live-tracker app: **Upload > Upload a video and its keypoints**, choose the two files, press *Use these files*.

## 1. Is there a GPU?

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
if not gpus:
    print("NO GPU. Use Runtime > Change runtime type > T4 GPU, then run this cell again.")

## 2. Get the script and the MARS models (543 MB, checked with md5)

In [ ]:
!wget -q -O mars_pose_offline.py https://raw.githubusercontent.com/AvaSajjadi/live-tracker/ava/demo-integration/scripts/mars_pose_offline.py
!python mars_pose_offline.py --download-only --models-dir mars_models

## 3. Choose your video
Use the file you recorded of the screen (or any top-view video of a black and a white mouse).
For a big file, mounting Google Drive is faster than uploading. Use **one** of the two options below.

In [ ]:
# Option A: upload from your computer
from google.colab import files
uploaded = files.upload()
video = next(iter(uploaded))
print("video:", video)

In [ ]:
# Option B: from Google Drive (remove the # signs and put your own path)
# from google.colab import drive
# drive.mount('/content/drive')
# video = '/content/drive/MyDrive/camera_recording.mp4'

## 4. Corners and a speed test
If the camera filmed a **screen** (for example an iPad), give the 4 corners of the video: on the Upload page of the app,
move the red dots on the live camera video, and copy the line of numbers shown there. If the video is already only the
mouse cage, leave `CORNERS` empty.

In [ ]:
CORNERS = ""   # for example "0.23,0.22,0.81,0.22,0.81,0.65,0.23,0.65"  (top-left, top-right, bottom-right, bottom-left)
BATCH = 16     # pictures processed together; if you see an out-of-memory error, use 8 or 4
corner_arg = f'--corners {CORNERS}' if CORNERS else ''
!python mars_pose_offline.py "{video}" --out test.parquet --models-dir mars_models --max-frames 60 --batch {BATCH} {corner_arg}

Look at the *frames per second* in the last lines above (the first batch is slower because the GPU is warming up).
Live use needs about 30 frames per second. If it is much lower, the full run will take (number of frames) / (frames per second).

## 5. The full run

In [ ]:
!python mars_pose_offline.py "{video}" --out keypoints.parquet --save-video stretched.mp4 --models-dir mars_models --batch {BATCH} {corner_arg}

## 6. Quick look: are the points on the mice?

In [ ]:
import cv2, pandas as pd
import matplotlib.pyplot as plt
df = pd.read_parquet("keypoints.parquet")
cap = cv2.VideoCapture("stretched.mp4")
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
last = int(df.frame.max())
for ax, f in zip(axes, [0, last // 2, last]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, img = cap.read()
    if not ok:
        continue
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for mouse, color in (("a", "red"), ("b", "cyan")):
        d = df[(df.frame == f) & (df.mouse == mouse)]
        ax.scatter(d.x, d.y, c=color, s=25)
    ax.set_title(f"frame {f}: red = mouse a (black), cyan = mouse b (white)")
    ax.axis("off")
plt.show()
print(df.groupby("mouse").score.describe()[["count", "mean", "min"]])

If the dots are on the mice, go on. If they are not, the picture probably does not look like the MARS training videos
(color of the mice, view from above, sharpness), and the model will not give good keypoints for it.

## 7. Download the two files

In [ ]:
from google.colab import files
files.download("keypoints.parquet")
files.download("stretched.mp4")